# Balanced Data Preparation & Advanced Augmentation



In [3]:
import os
import numpy as np
import torch
import torchvision.transforms.functional as TF
import torchvision.transforms as T
from tqdm import tqdm
from collections import Counter
from sklearn.model_selection import train_test_split
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
from utils.helper import set_seed, load_data, preprocess, create_patches

set_seed(42)
OUTPUT_DIR = "../processed_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)
PATCH_SIZE = 32
STRIDE = 8

In [4]:
print("Loading Raw Data\n")
ls_data, lulc_data = load_data(base_path="../landsat_data/", lulc_path="lulc.tif")
ls_scaled, ndvi, rgb_vis = preprocess(ls_data)

# Combine bands and NDVI
features = np.concatenate([ls_scaled.values, ndvi.values[None, ...]], axis=0)
labels = lulc_data.values
print(f"Feature stack shape : {features.shape}")

Loading Raw Data

Feature stack shape : (5, 205, 286)


### Extracting the Patches

In [5]:
print("Extracting Patches\n")
X_base, y_base, coords_base = create_patches(features, labels, patch_size=PATCH_SIZE, stride=STRIDE)

mid = PATCH_SIZE // 2
valid_mask = ~np.isnan(y_base[:, mid, mid])
X_base = X_base[valid_mask]
y_base = y_base[valid_mask]
coords_base = coords_base[valid_mask]

y_base = np.nan_to_num(y_base, nan=0.0)
unique_labels = np.sort(np.unique(y_base))
label_map = {lbl: i for i, lbl in enumerate(unique_labels)}
inv_label_map = {i: lbl for lbl, i in label_map.items()}
y_base_enc = np.vectorize(label_map.get)(y_base).astype(np.int64)

print(f"Extracted {len(X_base)} valid base patches.")

print("Splitting dataset into Train, Val, Test (60/20/20)")
mid = PATCH_SIZE // 2
X_train, X_tmp, y_train, y_tmp, c_train, c_tmp = train_test_split(X_base, y_base_enc, coords_base, test_size=0.40, random_state=42, stratify=y_base_enc[:, mid, mid])
X_val, X_test, y_val, y_test, c_val, c_test = train_test_split(X_tmp, y_tmp, c_tmp, test_size=0.5, random_state=42, stratify=y_tmp[:, mid, mid])
print(f"\nInitial Split - Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Extracting Patches

Extracted 913 valid base patches.
Splitting dataset into Train, Val, Test (60/20/20)

Initial Split - Train: 547 | Val: 183 | Test: 183


In [6]:
mid = PATCH_SIZE // 2
train_patch_classes = y_train[:, mid, mid]
counts = Counter(train_patch_classes)
max_class = counts.most_common(1)[0][0]
max_count = counts[max_class]

print("Training Set Class Distribution:")
for c in sorted(counts.keys()):
    print(f"Class {inv_label_map[c]} ({c}): {counts[c]} patches")

balancing_multipliers = {c: int(np.ceil(max_count / counts[c])) for c in counts}
balancing_multipliers[max_class] = 1

print("\nRequested Augmentation Multipliers (for training set only):")
for c in sorted(balancing_multipliers.keys()):
    print(f"Class {inv_label_map[c]} ({c}): x{balancing_multipliers[c]}")

Training Set Class Distribution:
Class 1.0 (1): 99 patches
Class 2.0 (2): 71 patches
Class 7.0 (3): 132 patches
Class 11.0 (4): 245 patches

Requested Augmentation Multipliers (for training set only):
Class 1.0 (1): x3
Class 2.0 (2): x4
Class 7.0 (3): x2
Class 11.0 (4): x1


### Advanced Augmentation

In [7]:
def augment_advanced(x_np, y_np, coord_np, multiplier):
    if multiplier == 1:
        return x_np[None, ...], y_np[None, ...], coord_np[None, ...]
    
    aug_x = [x_np]
    aug_y = [y_np]
    aug_c = [coord_np]
    
    xt = torch.from_numpy(x_np)
    yt = torch.from_numpy(y_np)

    for _ in range(multiplier - 1):
        x_curr, y_curr = xt.clone(), yt.clone()
        
        # Geometric
        k = np.random.randint(0, 4)
        x_curr = torch.rot90(x_curr, k, dims=[1, 2])
        y_curr = torch.rot90(y_curr, k, dims=[0, 1])
        if np.random.random() > 0.5:
            x_curr = torch.flip(x_curr, dims=[2])
            y_curr = torch.flip(y_curr, dims=[1])
            
        # Spectral
        brightness = 1.0 + (np.random.random() - 0.5) * 0.4
        x_curr = torch.clamp(x_curr * brightness, 0, 1)
        if np.random.random() > 0.7:
            x_curr = TF.gaussian_blur(x_curr, kernel_size=[3, 3], sigma=[0.1, 1.0])
        if np.random.random() > 0.5:
            noise = torch.randn_like(x_curr) * 0.01
            x_curr = torch.clamp(x_curr + noise, 0, 1)
            
        aug_x.append(x_curr.numpy())
        aug_y.append(y_curr.numpy())
        aug_c.append(coord_np) 
    return np.array(aug_x), np.array(aug_y), np.array(aug_c)

print("Applying balanced augmentation to TRAINING set\n")
X_train_final, y_train_final, c_train_final = [], [], []

for i in tqdm(range(len(X_train))):
    c_id = train_patch_classes[i]
    m = balancing_multipliers[c_id]
    ax, ay, ac = augment_advanced(X_train[i], y_train[i], c_train[i], m)
    X_train_final.append(ax)
    y_train_final.append(ay)
    c_train_final.append(ac)

X_train_final = np.concatenate(X_train_final, axis=0)
y_train_final = np.concatenate(y_train_final, axis=0)
c_train_final = np.concatenate(c_train_final, axis=0)

print(f"\nFinal augmented training set size : {len(X_train_final)} samples.")

Applying balanced augmentation to TRAINING set



100%|██████████| 547/547 [00:00<00:00, 599.00it/s] 



Final augmented training set size : 1090 samples.


### Saving the Data

In [8]:
np.save(os.path.join(OUTPUT_DIR, "X_train.npy"), X_train_final)
np.save(os.path.join(OUTPUT_DIR, "y_train.npy"), y_train_final)
np.save(os.path.join(OUTPUT_DIR, "c_train.npy"), c_train_final)

np.save(os.path.join(OUTPUT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(OUTPUT_DIR, "y_val.npy"), y_val)
np.save(os.path.join(OUTPUT_DIR, "c_val.npy"), c_val)

np.save(os.path.join(OUTPUT_DIR, "X_test.npy"), X_test)
np.save(os.path.join(OUTPUT_DIR, "y_test.npy"), y_test)
np.save(os.path.join(OUTPUT_DIR, "c_test.npy"), c_test)

np.save(os.path.join(OUTPUT_DIR, "unique_labels.npy"), unique_labels)

print(f"Successfully saved datasets to {OUTPUT_DIR} folder")
print("\nFinal Dataset Counts :")
print(f"Train (Augmented) : {len(X_train_final)}")
print(f"Validation Data : {len(X_val)} ")
print(f"Testing Data : {len(X_test)}")

Successfully saved datasets to ../processed_data folder

Final Dataset Counts :
Train (Augmented) : 1090
Validation Data : 183 
Testing Data : 183
